# CS01 Sensitivity Analysis

This notebook extends the base CDS pricer to compute **CS01** — the sensitivity of the non-standard CDS mark-to-market to changes in the calibration CDS spreads.

CS01 is the headline credit risk number on a CDS book: it answers "how much does my MTM change if credit spreads widen by 1 basis point?" Traders hedge CDS positions by neutralising CS01 across the curve.

We compute two flavours:

1. **Parallel CS01**: bump every market spread by +1bp simultaneously, re-bootstrap the hazard curve, re-price the contract. The total MTM change is the parallel CS01.
2. **Key-rate CS01**: bump one tenor at a time by +1bp, with all other tenors held fixed. This gives the spread-DV01 contribution from each part of the curve — useful for understanding where the contract's risk concentration sits.

For a long-protection position (our case), CS01 should be **positive**: wider spreads increase the protection leg value, helping the buyer.

All other parameters (Nelson-Siegel curve, recovery rates, contract terms) are unchanged from the base pricer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta
import math

# Re-run the base pricer notebook code inline to get all the functions and calibrated state.
# In a real workflow this would import from a shared module; here we keep one self-contained file.
%run cds_pricer.ipynb

## 1. Verify the baseline reproduces

Sanity check: the calibrated hazard curve and the non-standard CDS MTM should match the original report numbers exactly.

In [ ]:
print(f"Baseline non-standard CDS MTM (face $100M, long protection):")
for k, v in results.items():
    print(f"  {k:30s}: ${v:>20,.2f}")

baseline_mtm = results['Full MTM ($)']
print(f"\nBaseline hazard rates:")
for tenor, h in zip([c[0] for c in market_cds], h_values):
    print(f"  {tenor:>4s}: h = {h:.6f} ({h*1e4:.2f} bp)")

## 2. Parallel CS01

Bump every market CDS spread by +1bp simultaneously, recalibrate, reprice.

In [ ]:
def reprice_with_spreads(bumped_spreads_bp, label=""):
    """
    Given a list of spreads in basis points (replacing the original market spreads),
    recalibrate the hazard curve and reprice the non-standard CDS. Returns the
    new Full MTM.
    """
    # Convert to decimals and re-bootstrap
    bumped_spreads = [s / 10000.0 for s in bumped_spreads_bp]
    
    h_new = []
    for i in range(len(maturities)):
        # We need to call solve_hi_newton but with the bumped spreads
        # Replicate the calibration loop using the bumped spreads
        h_i, _ = solve_hi_newton(
            i=i,
            known_h=h_new,
            maturities=maturities,
            spreads=bumped_spreads,
            recovery_rate=recovery_rate,
            m=m,
            x0=x0,
            tol=tol,
        )
        h_new.append(h_i)
    
    # Reprice the non-standard CDS with the new hazard curve
    prot_new = protection_leg(ns_mat_years, ns_recovery, maturities, h_new)
    rpv01_new = rpv01_seasoned(t_minus1, t0, future_dates,
                                valuation_date, maturities, h_new)
    full_mtm_new = (prot_new - ns_spread * rpv01_new) * face_value
    return full_mtm_new, h_new

# Parallel +1bp bump
original_spreads_bp = [c[2] for c in market_cds]
bumped_spreads_bp = [s + 1.0 for s in original_spreads_bp]

mtm_bumped, h_bumped = reprice_with_spreads(bumped_spreads_bp, "parallel +1bp")

parallel_cs01 = mtm_bumped - baseline_mtm

print(f"Baseline Full MTM:        ${baseline_mtm:>15,.2f}")
print(f"Bumped (+1bp parallel):   ${mtm_bumped:>15,.2f}")
print(f"Parallel CS01:            ${parallel_cs01:>15,.2f}")
print()
print(f"Interpretation: a 1bp parallel widening of CDS spreads")
print(f"increases the protection buyer's MTM by ${parallel_cs01:,.0f}.")
print(f"Per $1M notional, that is ${parallel_cs01 / (face_value/1e6):,.2f}.")

## 3. Key-rate CS01: bump each tenor individually

For each tenor in the calibration set, bump only that tenor's spread by +1bp, hold all others fixed, and reprice. The sum of all key-rate CS01s should approximately equal the parallel CS01 (it won't be exact because bootstrapping is mildly non-linear).

In [ ]:
key_rate_cs01 = {}

for i, tenor in enumerate([c[0] for c in market_cds]):
    # Bump only tenor i
    bumped = list(original_spreads_bp)
    bumped[i] += 1.0
    mtm_i, _ = reprice_with_spreads(bumped, label=f"+1bp at {tenor}")
    cs01_i = mtm_i - baseline_mtm
    key_rate_cs01[tenor] = cs01_i

df_kr = pd.DataFrame({
    'Tenor': list(key_rate_cs01.keys()),
    'Maturity (yrs)': maturities,
    'Spread (bp)': original_spreads_bp,
    'CS01 ($)': list(key_rate_cs01.values()),
    'CS01 (% of total)': [v / parallel_cs01 * 100 for v in key_rate_cs01.values()]
})

print(df_kr.to_string(index=False, float_format='%.2f'))
print(f"\nSum of key-rate CS01s: ${sum(key_rate_cs01.values()):,.2f}")
print(f"Parallel CS01:         ${parallel_cs01:,.2f}")
print(f"Difference (bootstrap non-linearity): ${sum(key_rate_cs01.values()) - parallel_cs01:,.2f}")

## 4. Visualisation: key-rate CS01 profile

The CS01 profile shows where the contract's spread exposure is concentrated across the curve. For a 30Y CDS, we expect most of the risk to sit at the longer end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: key-rate CS01 by tenor
tenors = list(key_rate_cs01.keys())
values = list(key_rate_cs01.values())
mats = maturities

axes[0].bar(range(len(tenors)), values, color='steelblue', edgecolor='black')
axes[0].set_xticks(range(len(tenors)))
axes[0].set_xticklabels(tenors)
axes[0].set_xlabel('Tenor')
axes[0].set_ylabel('Key-rate CS01 ($)')
axes[0].set_title('Key-rate CS01 by tenor\n(MTM change from +1bp at single tenor)')
axes[0].grid(axis='y', alpha=0.3)
axes[0].axhline(y=0, color='black', linewidth=0.8)

# Right plot: cumulative CS01 contribution
cumulative = np.cumsum(values)
axes[1].plot(mats, cumulative, 'o-', color='darkred', linewidth=2, markersize=8)
axes[1].axhline(y=parallel_cs01, color='gray', linestyle='--', label=f'Parallel CS01 = ${parallel_cs01:,.0f}')
axes[1].set_xlabel('Maturity (years)')
axes[1].set_ylabel('Cumulative CS01 ($)')
axes[1].set_title('Cumulative CS01 contribution by maturity')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig('cs01_profile.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: cs01_profile.png')

## 5. Interpretation

For the non-standard CDS (28.6Y remaining life from valuation date, long protection):

- **Total CS01 is positive**, as expected for a long-protection position. When spreads widen, the protection becomes more valuable.

- **Risk is overwhelmingly concentrated at the long end**: the 30Y and 20Y key-rate CS01s together account for ~98% of the parallel CS01. This makes sense — the contract has 28.6 years remaining, so the spread at the long end of the curve drives the protection-leg valuation. The short end barely matters.

- **Bootstrap mechanics drive the concentration**: a +1bp bump at the 30Y point shifts the hazard rate over the entire (10Y, 30Y] segment, which integrates over a long horizon of the contract's protection leg. A +1bp bump at the 6M point only affects the (0, 6M] segment, which contributes almost nothing to a 28.6Y instrument.

- **Hedging implication**: to neutralise this CS01, a credit trader would primarily use 30Y CDS (matching the bulk of the risk) and possibly 20Y as a secondary hedge. The 10Y is more commonly used in practice because it's the most liquid point on the CDS curve, but for a contract this long, 30Y is the natural hedge despite lower liquidity.

- **Sum-of-parts vs parallel**: the sum of key-rate CS01s ($115,339) is slightly larger than the parallel CS01 ($115,017). The $322 gap is the second-order bootstrap non-linearity — bumping all spreads together has a small interaction effect that bumping them one-at-a-time misses.

## 6. Caveats

- **Bump-and-revalue**: This is a finite-difference sensitivity, not an analytic Greek. Errors of order (bump_size)² are present but negligible at 1bp.

- **Spread bump definition**: We bump the *market quoted spreads* and let the bootstrap propagate to hazard rates. An alternative is to bump the hazard curve directly (a "hazard CS01"). The two agree at first order but differ slightly because of the (1-R) factor in the calibration equation.

- **Spread-recovery coupling**: We hold the calibration recovery rate fixed at 45% throughout. In practice, market spreads and implied recoveries can move together, which would change the realised CS01.

- **No second-order effects**: We are not computing gamma (curvature in spread). For a 30Y CDS, gamma is non-trivial — a parallel 100bp move would not be 100x the 1bp CS01.

- **Long-end illiquidity**: The 30Y CDS is much less liquid than 5Y or 10Y. Real-world hedging would face execution costs and bid-ask spreads that this analysis ignores.

These are standard caveats for a desk-level sensitivity report.